# 03 — Threshold Selection & Cost Analysis

Two competing objectives:
1. **Cost-optimal threshold** — minimise total dollar loss (C_FN=$420, C_FP=$6)
2. **Capacity-constrained threshold** — lowest threshold where analyst queue ≤ 200 cases/day

Output: `src/fraud_detection_demo/demo/data_export.json`

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score

DATA_DIR  = Path("../../..") / "data" / "ieee-fraud-detection"
DEMO_DIR  = Path("..") / "demo"
EXPORT_PATH = DEMO_DIR / "data_export.json"

C_FN = 420.0   # cost of missed fraud: avg order value + chargeback penalty
C_FP = 6.0     # cost of false positive: review time + customer friction
CAPACITY_PER_DAY = 200
print(f"Cost FN=${C_FN}  FP=${C_FP}  |  imbalance ratio: {C_FN/C_FP:.0f}:1")


## 1. Load test predictions

In [ ]:
df = pd.read_parquet(DATA_DIR / "test_predictions.parquet")
y_true = df["isFraud"].values
y_prob = df["y_prob"].values

pr_auc = average_precision_score(y_true, y_prob)
print(f"Test set: {len(df):,} rows  |  fraud: {y_true.mean():.2%}")
print(f"PR-AUC: {pr_auc:.4f}")


## 2. Compute PR curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
thresholds = np.append(thresholds, 1.0)   # align lengths

print(f"PR curve points: {len(thresholds)}")
print(f"Recall range: {recall.min():.3f} – {recall.max():.3f}")
print(f"Precision range: {precision.min():.3f} – {precision.max():.3f}")


## 3. Cost sweep across all thresholds

In [ ]:
def compute_cost(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    return float(C_FN * fn + C_FP * fp)

costs          = np.array([compute_cost(y_true, y_prob, t) for t in thresholds])
flagged_counts = np.array([(y_prob >= t).sum() for t in thresholds])
flagged_per_day = flagged_counts / 30.0   # test set ≈ 30 days of traffic

best_idx = int(np.argmin(costs))
print(f"Cost-optimal threshold : {thresholds[best_idx]:.3f}")
print(f"  Precision: {precision[best_idx]:.3f}  Recall: {recall[best_idx]:.3f}")
print(f"  Total cost: ${costs[best_idx]:,.0f}  |  Flagged/day: {flagged_per_day[best_idx]:.0f}")


## 4. Capacity-constrained threshold

The cost-optimal threshold flags 1,200+ cases/day — far above analyst capacity. We find the lowest threshold (highest recall) where volume stays ≤ 200/day.

In [ ]:
cap_idx = next(i for i, fpd in enumerate(flagged_per_day) if fpd <= CAPACITY_PER_DAY)

print(f"Capacity-constrained threshold : {thresholds[cap_idx]:.3f}")
print(f"  Precision : {precision[cap_idx]:.3f}")
print(f"  Recall    : {recall[cap_idx]:.3f}")
print(f"  Cost/day  : ${costs[cap_idx]/30:,.0f}")
print(f"  Flags/day : {flagged_per_day[cap_idx]:.0f}")


## 5. Export PR curve + thresholds to JSON

In [ ]:
pr_curve = [
    {
        "threshold": round(float(t), 4),
        "precision": round(float(p), 4),
        "recall": round(float(r), 4),
        "cost": round(float(c), 0),
        "flagged_per_day": round(float(f), 1),
    }
    for t, p, r, c, f in zip(thresholds, precision, recall, costs, flagged_per_day)
]

if EXPORT_PATH.exists():
    with open(EXPORT_PATH) as f:
        export = json.load(f)
else:
    DEMO_DIR.mkdir(parents=True, exist_ok=True)
    export = {}

export.update({
    "pr_curve": pr_curve,
    "pr_auc": round(pr_auc, 4),
    "optimal_threshold": round(float(thresholds[cap_idx]), 4),
    "cost_optimal_threshold": round(float(thresholds[best_idx]), 4),
    "optimal_precision": round(float(precision[cap_idx]), 4),
    "optimal_recall": round(float(recall[cap_idx]), 4),
    "c_fn": C_FN,
    "c_fp": C_FP,
    "capacity_per_day": CAPACITY_PER_DAY,
})

with open(EXPORT_PATH, "w") as f:
    json.dump(export, f)

print(f"Exported {len(pr_curve)} PR curve points → {EXPORT_PATH}")
